In [1]:
!pip install -q datasets transformers

In [2]:
import torch
import re
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# CONFIG
# =========================

MODEL_NAME = "microsoft/phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_NEW_TOKENS = 256
NUM_SAMPLES = 50
NUM_GENERATIONS = 5   # self-consistency boost

# =========================
# LOAD MODEL
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto"
)

model.eval()

# =========================
# PROMPT (HIGH PERFORMANCE)
# =========================

def build_prompt(question, task="math"):
    if task == "math":
        content = f"""
Solve this step-by-step carefully. Verify each step.

Give final answer clearly at the end.

Question: {question}
"""
    elif task == "qa":
        content = f"""
Answer yes or no with reasoning.

Question: {question}
"""
    elif task == "mcq":
        content = f"""
Choose the correct option (A/B/C/D). Think step-by-step.

{question}
"""
    else:
        content = question

    messages = [
        {"role": "system", "content": "You are a careful reasoning assistant."},
        {"role": "user", "content": content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

# =========================
# UTILS
# =========================

def extract_number(text):
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
    return nums[-1] if nums else None

def extract_yes_no(text):
    text = text.lower()
    if "yes" in text:
        return "yes"
    if "no" in text:
        return "no"
    return None

def extract_mcq(text):
    text = text.upper()
    for c in ["A", "B", "C", "D"]:
        if f" {c}" in text or text.strip().startswith(c):
            return c
    return None

def majority_vote(preds):
    preds = [p for p in preds if p is not None]
    if not preds:
        return None
    return max(set(preds), key=preds.count)

# =========================
# GENERATION
# =========================

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

def multi_generate(prompt, extractor):
    preds = []
    for _ in range(NUM_GENERATIONS):
        out = generate(prompt)
        preds.append(extractor(out))
    return preds

# =========================
# GSM8K
# =========================

def run_gsm8k(dataset):
    results = []

    for sample in tqdm(dataset):
        q = sample["question"]
        gt = sample["answer"].split("####")[-1].strip()

        prompt = build_prompt(q, "math")
        preds = multi_generate(prompt, extract_number)

        final = majority_vote(preds)

        correct = False
        try:
            correct = float(final) == float(gt)
        except:
            pass

        results.append(correct)

    acc = sum(results) / len(results)
    print("\nGSM8K Accuracy:", acc)
    return acc

# =========================
# STRATEGYQA
# =========================

def run_strategyqa(dataset):
    results = []

    for sample in tqdm(dataset):
        q = sample["question"]
        gt = "yes" if sample["answer"] else "no"

        prompt = build_prompt(q, "qa")
        preds = multi_generate(prompt, extract_yes_no)

        final = majority_vote(preds)

        results.append(final == gt)

    acc = sum(results) / len(results)
    print("\nStrategyQA Accuracy:", acc)
    return acc

# =========================
# MMLU
# =========================

def run_mmlu(dataset):
    results = []

    for sample in tqdm(dataset):
        q = sample["question"]
        choices = sample["choices"]
        gt = chr(65 + sample["answer"])

        formatted_q = q + "\n"
        for i, c in enumerate(choices):
            formatted_q += f"{chr(65+i)}. {c}\n"

        prompt = build_prompt(formatted_q, "mcq")
        preds = multi_generate(prompt, extract_mcq)

        final = majority_vote(preds)

        results.append(final == gt)

    acc = sum(results) / len(results)
    print("\nMMLU Accuracy:", acc)
    return acc

# =========================
# LOAD DATASETS
# =========================

gsm8k = load_dataset("gsm8k", "main", split="test").select(range(NUM_SAMPLES))
strategyqa = load_dataset("ChilleD/StrategyQA", split="test").select(range(NUM_SAMPLES))
mmlu = load_dataset("cais/mmlu", "abstract_algebra", split="test").select(range(NUM_SAMPLES))

# =========================
# RUN ALL
# =========================

gsm_acc = run_gsm8k(gsm8k)
strat_acc = run_strategyqa(strategyqa)
mmlu_acc = run_mmlu(mmlu)

print("\nFINAL RESULTS")
print("GSM8K:", gsm_acc)
print("StrategyQA:", strat_acc)
print("MMLU:", mmlu_acc)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...


GSM8K Accuracy: 0.38


100%|██████████| 50/50 [19:48<00:00, 23.77s/it]



StrategyQA Accuracy: 0.42


100%|██████████| 50/50 [36:31<00:00, 43.82s/it]


MMLU Accuracy: 0.2

FINAL RESULTS
GSM8K: 0.38
StrategyQA: 0.42
MMLU: 0.2
